# PLUTO: from vector scene to trajectory

This offline lesson implements a **simplified educational variant**, with randomly initialized weights and toy synthetic data. It does not reproduce nuPlan or the official code. See `docs/paper-to-code.md` in the repository.




In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(7)
torch.set_num_threads(1)


## Data contract

Masks use `True` for valid points. The first agent is ego; positions are metres in the ego frame. This synthetic fixture supplies mode labels directly; an elementary projection helper is available but is not connected to real preprocessing. Shapes below use batch size 2 and a four-step horizon.




In [ ]:
"""Deterministic synthetic input fixture, not a nuPlan adapter."""



def sample_batch(batch_size=2, future_steps=4, seed=42):
    g = torch.Generator().manual_seed(seed)

    def rand(*shape):
        return torch.randn(*shape, generator=g)

    b, a, r = batch_size, 3, 2
    target = rand(b, future_steps, 6) * 0.1
    return {
        "agents": rand(b, a, 5, 9),
        "agent_mask": torch.ones(b, a, 5, dtype=torch.bool),
        "ego_state": rand(b, 6),
        "map": rand(b, 3, 5, 10),
        "map_mask": torch.ones(b, 3, 5, dtype=torch.bool),
        "static": rand(b, 2, 5),
        "static_mask": torch.ones(b, 2, dtype=torch.bool),
        "reference": rand(b, r, 6, 6),
        "reference_mask": torch.ones(b, r, 6, dtype=torch.bool),
        "target": target,
        "target_mask": torch.ones(b, future_steps, dtype=torch.bool),
        "agent_target": rand(b, a - 1, future_steps, 6) * 0.1,
        "agent_target_mask": torch.ones(b, a - 1, future_steps, dtype=torch.bool),
        "mode_label": torch.zeros(b, dtype=torch.long),
    }

batch = sample_batch(batch_size=2, future_steps=4)
print({k: tuple(v.shape) for k, v in batch.items()})


## Masked history pyramid

The history encoder applies local self-attention at three temporal scales, combines the scales with lateral projections, and reads the last valid original timestep. The windows and mask behavior are tested, but exact NATTEN operator parity is not established.




## Architecture

Map point embeddings are pooled over valid elements; map points include left/right lane boundaries and semantic attributes. Ego scalar features use attention with optional dropout. Queries combine one reference-line token and one learned longitudinal mode. Decoder layers attend across references, then modes, then encoded scene tokens. Empty references use the free head.




In [ ]:
"""Vector scene encoder and factorized lateral/longitudinal planner.

All spatial inputs are in the ego frame, in metres. Masks are True for valid
entries. The first agent is ego; other agent future predictions are agent-relative.
"""



@dataclass
class PlutoConfig:
    dim: int = 128
    heads: int = 4
    encoder_layers: int = 4
    decoder_layers: int = 4
    modes: int = 12
    future_steps: int = 80
    dropout: float = 0.1
    state_dropout: float = 0.75


def mlp(n_in, hidden, n_out):
    return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, n_out))


class PointEncoder(nn.Module):
    """Masked pointwise MLP and max pooling; empty polylines map to zero."""

    def __init__(self, channels, dim):
        super().__init__()
        self.net = mlp(channels, dim, dim)

    def forward(self, x, mask):
        y = self.net(x).masked_fill(~mask.unsqueeze(-1), float("-inf"))
        pooled = y.max(dim=-2).values
        return torch.where(
            mask.any(dim=-1, keepdim=True), pooled, torch.zeros_like(pooled)
        )


class LocalAttentionBlock(nn.Module):
    """Windowed temporal attention with a residual MLP and valid-key masks."""

    def __init__(self, dim, heads, window=3):
        super().__init__()
        if window < 1 or window % 2 != 1:
            raise ValueError("window must be a positive odd integer")
        self.window = window
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.feedforward = nn.Sequential(
            nn.Linear(dim, dim * 3), nn.GELU(), nn.Linear(dim * 3, dim)
        )

    def forward(self, x, valid):
        length = x.shape[1]
        offsets = torch.arange(length, device=x.device)
        outside = (offsets[:, None] - offsets[None, :]).abs() > self.window // 2
        safe_valid = valid.clone()
        safe_valid[~safe_valid.any(-1), 0] = True
        # A valid key may lie outside a padded query's window; grant padded
        # queries one dummy key, then zero their outputs immediately afterwards.
        expanded = outside.unsqueeze(0).expand(x.shape[0], -1, -1).clone()
        expanded |= ~safe_valid[:, None, :]
        empty = expanded.all(-1)
        expanded[empty] = False
        mask = expanded.repeat_interleave(self.attention.num_heads, dim=0)
        x = x.masked_fill(~valid.unsqueeze(-1), 0.0)
        value = self.norm1(x)
        x = (
            x
            + self.attention(value, value, value, attn_mask=mask, need_weights=False)[0]
        )
        x = x + self.feedforward(self.norm2(x))
        return x.masked_fill(~valid.unsqueeze(-1), 0.0)


class HistoryEncoder(nn.Module):
    """Three-level local-attention pyramid for masked agent history vectors.

    Window boundaries and masking are implemented in PyTorch; this is not yet
    operator-compatible with the released NATTEN implementation.
    """

    def __init__(self, channels, dim, heads):
        super().__init__()
        if dim % 4:
            raise ValueError("history embedding dim must be divisible by four")
        dims = (dim // 4, dim // 2, dim)
        self.tokenizer = nn.Conv1d(channels, dims[0], 3, padding=1)
        self.levels = nn.ModuleList(
            [
                nn.ModuleList(
                    [
                        LocalAttentionBlock(
                            width, math.gcd(width, heads), window=3 if i < 2 else 5
                        )
                        for _ in range(2)
                    ]
                )
                for i, width in enumerate(dims)
            ]
        )
        self.down = nn.ModuleList(
            [nn.Conv1d(dims[i], dims[i + 1], 3, stride=2, padding=1) for i in range(2)]
        )
        self.lateral = nn.ModuleList(
            [nn.Conv1d(width, dim, 3, padding=1) for width in dims]
        )
        self.fuse = nn.Conv1d(dim, dim, 3, padding=1)

    def forward(self, history, valid):
        b, a, t, c = history.shape
        mask = valid.reshape(b * a, t)
        value = history.reshape(b * a, t, c).masked_fill(~mask.unsqueeze(-1), 0.0)
        x = self.tokenizer(value.transpose(1, 2)).transpose(1, 2)
        outputs = []
        for i, blocks in enumerate(self.levels):
            for block in blocks:
                x = block(x, mask)
            outputs.append(x.transpose(1, 2))
            if i < len(self.down):
                x = self.down[i](x.transpose(1, 2)).transpose(1, 2)
                mask = (
                    F.max_pool1d(mask.float().unsqueeze(1), 3, stride=2, padding=1)
                    .squeeze(1)
                    .bool()
                )
        levels = [layer(value) for layer, value in zip(self.lateral, outputs)]
        for i in range(len(levels) - 1, 0, -1):
            levels[i - 1] = levels[i - 1] + F.interpolate(
                levels[i],
                size=levels[i - 1].shape[-1],
                mode="linear",
                align_corners=False,
            )
        features = self.fuse(levels[0]).transpose(1, 2)
        original = valid.reshape(b * a, t)
        last = (
            torch.arange(t, device=history.device)
            .expand(b * a, t)
            .masked_fill(~original, -1)
            .max(-1)
            .values
        )
        result = features[torch.arange(b * a, device=history.device), last.clamp(min=0)]
        result = result.masked_fill((last < 0).unsqueeze(-1), 0.0)
        return result.reshape(b, a, -1)


class StateEncoder(nn.Module):
    """Attend to ego-state scalars; randomly hide dynamic scalars during training."""

    def __init__(self, dim, heads, dropout=0.75):
        super().__init__()
        if not 0 <= dropout <= 1:
            raise ValueError("state dropout must be in [0,1]")
        self.dropout = dropout
        self.embed = nn.ModuleList([nn.Linear(1, dim) for _ in range(6)])
        self.position = nn.Parameter(torch.randn(1, 6, dim) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, state):
        values = (
            torch.stack(
                [layer(state[:, i : i + 1]) for i, layer in enumerate(self.embed)],
                dim=1,
            )
            + self.position
        )
        mask = torch.zeros(state.shape[0], 6, dtype=torch.bool, device=state.device)
        if self.training and self.dropout:
            mask[:, 3:] = (
                torch.rand(state.shape[0], 3, device=state.device) < self.dropout
            )
        return self.attention(
            self.query.expand(state.shape[0], -1, -1),
            values,
            values,
            key_padding_mask=mask,
            need_weights=False,
        )[0][:, 0]


class FourierPosition(nn.Module):
    """Learned Fourier features for ego-frame (x, y, heading) poses."""

    def __init__(self, dim, bands=16):
        super().__init__()
        self.frequency = nn.Parameter(torch.randn(3, bands))
        self.project = mlp(3 * (2 * bands + 1), dim, dim)

    def forward(self, pose):
        phase = pose.unsqueeze(-1) * self.frequency * (2 * math.pi)
        features = torch.cat((pose.unsqueeze(-1), phase.sin(), phase.cos()), dim=-1)
        return self.project(features.flatten(-2))


class DecoderLayer(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.lateral = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.longitudinal = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.cross = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(4)])
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, q, memory, ref_valid, scene_valid):
        b, r, m, d = q.shape
        x = q.transpose(1, 2).reshape(b * m, r, d)
        # An all-masked row yields NaNs in attention; a temporary dummy is ignored later.
        valid = ref_valid.clone()
        valid[~valid.any(-1), 0] = True
        mask = (~valid)[:, None, :].expand(b, m, r).reshape(b * m, r)
        y = self.norms[0](x)
        x = x + self.drop(
            self.lateral(y, y, y, key_padding_mask=mask, need_weights=False)[0]
        )
        x = x.reshape(b, m, r, d).transpose(1, 2).reshape(b * r, m, d)
        y = self.norms[1](x)
        x = x + self.drop(self.longitudinal(y, y, y, need_weights=False)[0])
        x = x.reshape(b, r * m, d)
        y = self.norms[2](x)
        x = x + self.drop(
            self.cross(
                y, memory, memory, key_padding_mask=~scene_valid, need_weights=False
            )[0]
        )
        x = x + self.drop(self.ff(self.norms[3](x)))
        return x.reshape(b, r, m, d)


class Pluto(nn.Module):
    """PLUTO-style model. See docs/paper-to-code.md for deliberate differences."""

    def __init__(self, config=PlutoConfig()):
        super().__init__()
        self.config = config
        d = config.dim
        self.agent_encoder = HistoryEncoder(9, d, config.heads)
        self.map_encoder = PointEncoder(10, d)
        self.static_encoder = mlp(5, d, d)
        self.ego_encoder = StateEncoder(d, config.heads, config.state_dropout)
        self.type_embedding = nn.Embedding(3, d)
        self.agent_type = nn.Embedding(4, d)
        self.map_type = nn.Embedding(3, d)
        self.route_type = nn.Embedding(2, d)
        self.traffic_type = nn.Embedding(4, d)
        self.static_type = nn.Embedding(4, d)
        self.speed_embedding = mlp(1, d, d)
        self.unknown_speed = nn.Parameter(torch.zeros(d))
        self.scene_position = FourierPosition(d)
        self.reference_position = FourierPosition(d)
        layer = nn.TransformerEncoderLayer(
            d, config.heads, d * 4, config.dropout, batch_first=True, norm_first=True
        )
        self.scene_encoder = nn.TransformerEncoder(
            layer, config.encoder_layers, norm=nn.LayerNorm(d)
        )
        self.reference_encoder = PointEncoder(6, d)
        self.mode = nn.Parameter(torch.randn(config.modes, d) * 0.01)
        self.query_projection = nn.Linear(2 * d, d)
        self.decoder = nn.ModuleList(
            [
                DecoderLayer(d, config.heads, config.dropout)
                for _ in range(config.decoder_layers)
            ]
        )
        self.trajectory_head = mlp(d, 2 * d, config.future_steps * 6)
        self.score_head = mlp(d, d, 1)
        self.prediction_head = mlp(d, 2 * d, config.future_steps * 6)
        self.free_head = mlp(d, 2 * d, config.future_steps * 6)
        self.projection = mlp(d, d, d)

    def encode_scene(self, batch):
        """Encode scene tokens; contrastive negatives need no trajectory decoder."""
        agents, maps, static = batch["agents"], batch["map"], batch["static"]
        b, a = agents.shape[:2]
        agent_valid = batch["agent_mask"].any(-1)
        # The ego state supplies a valid scene token even if its history is absent.
        agent_valid = agent_valid.clone()
        agent_valid[:, 0] = True
        agent = self.agent_encoder(agents, batch["agent_mask"])
        agent = torch.cat(
            [self.ego_encoder(batch["ego_state"]).unsqueeze(1), agent[:, 1:]], dim=1
        )
        map_count, static_count = maps.shape[1], static.shape[1]
        default_agents = torch.zeros(b, a, dtype=torch.long, device=agents.device)
        default_map = torch.zeros(b, map_count, dtype=torch.long, device=maps.device)
        default_static = torch.zeros(
            b, static_count, dtype=torch.long, device=static.device
        )
        agent = agent + self.agent_type(batch.get("agent_type", default_agents).long())
        map_embedding = (
            self.map_encoder(maps, batch["map_mask"])
            + self.map_type(batch.get("map_type", default_map).long())
            + self.route_type(batch.get("on_route", default_map).long())
            + self.traffic_type(batch.get("traffic_light", default_map).long())
        )
        speed = batch.get("speed_limit", maps.new_zeros(b, map_count))
        known = batch.get(
            "speed_known",
            torch.zeros(b, map_count, dtype=torch.bool, device=maps.device),
        )
        map_embedding = map_embedding + torch.where(
            known.unsqueeze(-1),
            self.speed_embedding(speed.unsqueeze(-1)),
            self.unknown_speed,
        )
        static_embedding = self.static_encoder(static) + self.static_type(
            batch.get("static_type", default_static).long()
        )
        tokens = torch.cat(
            [
                agent + self.type_embedding.weight[0],
                map_embedding + self.type_embedding.weight[1],
                static_embedding + self.type_embedding.weight[2],
            ],
            dim=1,
        )
        scene_valid = torch.cat(
            [agent_valid, batch["map_mask"].any(-1), batch["static_mask"]], dim=1
        )
        agent_pose = batch.get("agent_pose", agents.new_zeros(b, a, 3))
        map_pose = batch.get("map_pose", maps.new_zeros(b, maps.shape[1], 3))
        static_pose = batch.get("static_pose", static[..., :3])
        pose = torch.cat((agent_pose, map_pose, static_pose), dim=1)
        tokens = tokens + self.scene_position(pose)
        memory = self.scene_encoder(tokens, src_key_padding_mask=~scene_valid)
        return memory, scene_valid

    def forward(self, batch):
        c = self.config
        b, a = batch["agents"].shape[:2]
        memory, scene_valid = self.encode_scene(batch)
        prediction = self.prediction_head(memory[:, 1:a]).reshape(
            b, a - 1, c.future_steps, 6
        )
        free = self.free_head(memory[:, 0]).reshape(b, c.future_steps, 6)
        refs, ref_mask = batch["reference"], batch["reference_mask"]
        r = refs.shape[1]
        if r == 0:
            return {
                "trajectory": None,
                "logits": None,
                "prediction": prediction,
                "free": free,
                "selected": free,
                "hidden": self.projection(memory[:, 0]),
            }
        ref_valid = ref_mask.any(-1)
        ref_pose = batch.get("reference_pose", refs.new_zeros(b, r, 3))
        ref = self.reference_encoder(refs, ref_mask) + self.reference_position(ref_pose)
        q = self.query_projection(
            torch.cat(
                [
                    ref.unsqueeze(2).expand(b, r, c.modes, c.dim),
                    self.mode[None, None].expand(b, r, c.modes, c.dim),
                ],
                -1,
            )
        )
        for layer in self.decoder:
            q = layer(q, memory, ref_valid, scene_valid)
        trajectory = self.trajectory_head(q).reshape(b, r, c.modes, c.future_steps, 6)
        logits = (
            self.score_head(q).squeeze(-1).masked_fill(~ref_valid.unsqueeze(-1), -1e9)
        )
        best = logits.reshape(b, -1).argmax(-1)
        selected = trajectory.reshape(b, r * c.modes, c.future_steps, 6)[
            torch.arange(b, device=best.device), best
        ]
        selected = torch.where(ref_valid.any(-1)[:, None, None], selected, free)
        return {
            "trajectory": trajectory,
            "logits": logits,
            "prediction": prediction,
            "free": free,
            "selected": selected,
            "hidden": self.projection(memory[:, 0]),
        }


## Assemble and inspect

The toy network uses D=16 and one layer per stack, unlike the paper configuration (D=128, four layers and 12 longitudinal queries).




In [ ]:
cfg = PlutoConfig(dim=16, heads=4, encoder_layers=1, decoder_layers=1, modes=3, future_steps=4, dropout=0.0)
model = Pluto(cfg)
out = model(batch)
print('plans', out['trajectory'].shape, 'other agents', out['prediction'].shape)


## Objectives and optimization

The toy imitation loss uses a supplied reference/mode label, Smooth L1 trajectory and prediction terms and classification. The paper also trains with collision/off-road and contrastive terms; the standalone utilities below are not wired into this toy training step. SDF lookup is differentiable bilinear interpolation.




In [ ]:
"""Planning, agent prediction, contrastive and differentiable SDF objectives."""



def masked_smooth_l1(pred, target, valid):
    errors = F.smooth_l1_loss(pred, target, reduction="none").sum(-1)
    return (errors * valid.to(errors.dtype)).sum() / valid.sum().clamp(min=1)


def contrastive_loss(anchor, positive, negative, temperature=0.1, valid=None):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    a, p, n = (F.normalize(x, dim=-1) for x in (anchor, positive, negative))
    logits = torch.stack([(a * p).sum(-1), (a * n).sum(-1)], -1) / temperature
    loss = F.cross_entropy(
        logits,
        torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device),
        reduction="none",
    )
    if valid is None:
        return loss.mean() if loss.numel() else logits.sum() * 0
    return (loss * valid).sum() / valid.sum().clamp(min=1)


def sdf_loss(
    trajectory,
    sdf,
    resolution=0.2,
    offsets=(-1.0, 0.0, 1.0),
    radius=1.0,
    margin=0.0,
    valid=None,
):
    """Eq. 12: average sum of hinge penalties per timestep; x right, y up.

    SDF is positive in safe space, shape [B,H,W], with its origin at the
    center pixel. Out-of-bounds positions get zero distance (conservative).
    """
    if resolution <= 0:
        raise ValueError("resolution must be positive")
    h, w = sdf.shape[-2:]
    if h < 2 or w < 2:
        raise ValueError("SDF dimensions must be at least 2")
    off = trajectory.new_tensor(offsets).view(1, 1, -1, 1)
    centers = trajectory[..., None, :2] + off * trajectory[..., None, 2:4]
    gx = 2 * (centers[..., 0] / resolution + (w - 1) / 2) / (w - 1) - 1
    gy = 2 * (-centers[..., 1] / resolution + (h - 1) / 2) / (h - 1) - 1
    grid = torch.stack((gx, gy), -1)
    distance = F.grid_sample(
        sdf.unsqueeze(1),
        grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=True,
    ).squeeze(1)
    cost = (radius + margin - distance).clamp(min=0).sum(-1)
    if valid is None:
        return cost.mean()
    return (cost * valid.to(cost.dtype)).sum() / valid.sum().clamp(min=1)


def imitation_loss(out, batch):
    target, valid = batch["target"], batch["target_mask"]
    free = masked_smooth_l1(out["free"], target, valid)
    pred = masked_smooth_l1(
        out["prediction"], batch["agent_target"], batch["agent_target_mask"]
    )
    if out["trajectory"] is None:
        zero = free * 0
        return {
            "total": free + pred,
            "regression": zero,
            "classification": zero,
            "free": free,
            "prediction": pred,
        }
    traj, logits = out["trajectory"], out["logits"]
    b, r, m, t, _ = traj.shape
    ref_valid = batch["reference_mask"].any(-1)
    eligible = ref_valid.any(-1) & valid.any(-1)
    labels = batch["mode_label"].long()
    if (eligible & ((labels < 0) | (labels >= r * m))).any():
        raise ValueError("mode_label is outside the candidate range")
    idx = labels.clamp(0, r * m - 1)
    selected_valid = ref_valid.gather(1, (idx // m).unsqueeze(1)).squeeze(1)
    if (eligible & ~selected_valid).any():
        raise ValueError("mode_label refers to a masked reference line")
    chosen = traj.reshape(b, r * m, t, 6)[torch.arange(b, device=idx.device), idx]
    reg = masked_smooth_l1(chosen, target, valid & eligible[:, None])
    cls = (
        F.cross_entropy(logits.reshape(b, -1)[eligible], idx[eligible])
        if eligible.any()
        else logits.sum() * 0
    )
    return {
        "total": reg + cls + free + pred,
        "regression": reg,
        "classification": cls,
        "free": free,
        "prediction": pred,
    }


def assign_modes(endpoint, reference_xy, reference_mask, modes):
    """Project [B,2] endpoints onto valid polylines, returning flattened R*M labels.

    Longitudinal bins divide each reference length into M-1 equal intervals;
    the last bin is for projections at or beyond the end. Points are in metres
    in the same coordinate frame. Every sample needs a non-degenerate line.
    """
    if modes < 2 or reference_xy.shape[-2] < 2:
        raise ValueError("need at least two modes and two points per reference")
    start, end = reference_xy[..., :-1, :], reference_xy[..., 1:, :]
    delta = end - start
    length2 = (delta * delta).sum(-1)
    valid = reference_mask[..., :-1] & reference_mask[..., 1:] & (length2 > 1e-12)
    if not valid.any(dim=(-1, -2)).all():
        raise ValueError("every sample needs a valid, non-degenerate reference segment")
    length = length2.sqrt()
    offset = endpoint[:, None, None, :] - start
    u = ((offset * delta).sum(-1) / length2.clamp(min=1e-12)).clamp(0.0, 1.0)
    projected = start + u[..., None] * delta
    distance = (
        ((endpoint[:, None, None, :] - projected) ** 2)
        .sum(-1)
        .masked_fill(~valid, float("inf"))
    )
    b, r, s = distance.shape
    segment_index = distance.reshape(b, -1).argmin(-1)
    ref_index, point_index = segment_index // s, segment_index % s
    lengths = torch.where(valid, length, torch.zeros_like(length))
    accumulated = lengths.cumsum(-1) - lengths
    batch_idx = torch.arange(b, device=endpoint.device)
    arc = (
        accumulated[batch_idx, ref_index, point_index]
        + u[batch_idx, ref_index, point_index]
        * length[batch_idx, ref_index, point_index]
    )
    total = lengths.sum(-1)[batch_idx, ref_index]
    longitudinal = (
        torch.floor(arc / (total / (modes - 1)).clamp(min=1e-12))
        .long()
        .clamp(0, modes - 1)
    )
    return ref_index * modes + longitudinal


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = imitation_loss(out, batch)
optimizer.zero_grad()
losses['total'].backward()
optimizer.step()
print({k: round(v.item(), 4) for k, v in losses.items()})


## Contrastive imitation

A valid positive view preserves the target; a negative view changes a relevant cause, so its old target is not used. All three are encoded, but only original and positive views are decoded for supervision. The example is synthetic and its causal meaning is not validated.




In [ ]:
"""Contrastive imitation objective (paper Eq. 14) for prepared triplets."""



def _supervised_trajectory(out, batch):
    trajectory = out["trajectory"]
    valid = batch["target_mask"]
    if trajectory is None:
        return out["free"], valid
    b, r, m, t, channels = trajectory.shape
    labels = batch["mode_label"].long()
    ref_valid = batch["reference_mask"].any(-1)
    eligible = ref_valid.any(-1) & valid.any(-1)
    # imitation_loss checks label validity; fallback rows are not supervised here.
    chosen = trajectory.reshape(b, r * m, t, channels)[
        torch.arange(b, device=labels.device), labels.clamp(0, r * m - 1)
    ]
    return chosen, valid & eligible[:, None]


def supervised_objective(out, batch, sdf=None, resolution=0.2):
    """Imitation, agent prediction, and an optional labeled-plan SDF penalty."""
    losses = imitation_loss(out, batch)
    auxiliary = losses["total"] * 0
    if sdf is not None:
        trajectory, valid = _supervised_trajectory(out, batch)
        auxiliary = sdf_loss(trajectory, sdf, resolution=resolution, valid=valid)
    return {**losses, "auxiliary": auxiliary, "total": losses["total"] + auxiliary}


def contrastive_objective(
    model,
    original,
    positive,
    negative,
    sdf=None,
    positive_sdf=None,
    resolution=0.2,
    temperature=0.1,
    contrast_valid=None,
):
    """Supervise original/positive; negative contributes only to Eq. 13.

    Views are prepared externally: augmentations must preserve the positive
    target and invalidate the negative target for the actual scene. This
    function does not assert that a proposed augmentation is causally valid.
    SDF uses positive values for safe space; masks exclude invalid target steps.
    """
    anchor, augmented = model(original), model(positive)
    negative_memory, _ = model.encode_scene(negative)
    negative_hidden = model.projection(negative_memory[:, 0])
    orig_loss, pos_loss = (
        imitation_loss(anchor, original),
        imitation_loss(augmented, positive),
    )
    supervised = (orig_loss["total"] + pos_loss["total"]) * 0.5
    contrast = contrastive_loss(
        anchor["hidden"],
        augmented["hidden"],
        negative_hidden,
        temperature=temperature,
        valid=contrast_valid,
    )
    auxiliary = supervised * 0
    if sdf is not None:
        traj, mask = _supervised_trajectory(anchor, original)
        plus_traj, plus_mask = _supervised_trajectory(augmented, positive)
        plus_sdf = sdf if positive_sdf is None else positive_sdf
        auxiliary = (
            sdf_loss(traj, sdf, resolution=resolution, valid=mask)
            + sdf_loss(plus_traj, plus_sdf, resolution=resolution, valid=plus_mask)
        ) * 0.5
    return {
        "total": supervised + contrast + auxiliary,
        "supervised": supervised,
        "contrastive": contrast,
        "auxiliary": auxiliary,
    }


In [ ]:
positive = {k: v.clone() for k, v in batch.items()}
negative = {k: v.clone() for k, v in batch.items()}
negative['agent_mask'][:, -1] = False
cil_loss = contrastive_objective(model, batch, positive, negative)
optimizer.zero_grad()
cil_loss['total'].backward()
optimizer.step()
print({k: round(v.item(), 4) for k, v in cil_loss.items()})


## Inference and evaluation

Learned logits select a candidate. The paper additionally uses an official simulation-based scoring system, which is not replicated here. This average displacement error (ADE) is a synthetic open-loop diagnostic, not a nuPlan score.




In [ ]:
model.eval()
with torch.no_grad():
    pred = model(batch)['selected']
    ade = (pred[..., :2] - batch['target'][..., :2]).norm(dim=-1).mean()
print('synthetic ADE (m):', float(ade))


## Bicycle rollout and tracking

The reference plan is tracked with a finite-horizon linear-quadratic controller and a kinematic bicycle. The 0.1-second, straight-line example is analytically checkable. This illustrative tracker has not been compared with the nuPlan/PDM implementation or official safety rules.




In [ ]:
"""Educational batched kinematic bicycle and finite-horizon LQR tracking.

States are [x,y,yaw,speed], controls are [longitudinal acceleration, steering
angle], in metres, radians and seconds. This simulator is independent of the
nuPlan/PDM implementation and is not validated as an official replacement.
"""



def _wrap(angle):
    return torch.atan2(angle.sin(), angle.cos())


def bicycle_step(state, control, dt=0.1, wheelbase=2.7):
    if dt <= 0 or wheelbase <= 0:
        raise ValueError("dt and wheelbase must be positive")
    x, y, yaw, speed = state.unbind(-1)
    accel, steer = control.unbind(-1)
    return torch.stack(
        (
            x + speed * yaw.cos() * dt,
            y + speed * yaw.sin() * dt,
            _wrap(yaw + speed * steer.tan() * dt / wheelbase),
            (speed + accel * dt).clamp(min=0.0),
        ),
        dim=-1,
    )


def lqr_rollout(initial, trajectory, dt=0.1, wheelbase=2.7):
    """Track [B,K,T,6] plans from [B,4], returning [B,K,T+1,4] states.

    A finite-horizon LQR is computed around the trajectory's kinematic
    linearization using fixed diagonal state/control costs. Feedforward controls
    are inferred from reference velocity and yaw, then bounded to a plausible
    range. This diagnostic implementation does not model actuator lag or
    replicate the official PDM tracker.
    """
    if dt <= 0 or wheelbase <= 0 or trajectory.ndim != 4 or trajectory.shape[-1] < 6:
        raise ValueError("invalid trajectory, dt or wheelbase")
    b, k, horizon, _ = trajectory.shape
    if initial.shape != (b, 4) or horizon < 1:
        raise ValueError("initial must be [B,4] and horizon must be nonzero")
    yaw = torch.atan2(trajectory[..., 3], trajectory[..., 2])
    speed = torch.linalg.vector_norm(trajectory[..., 4:6], dim=-1)
    future = torch.cat(
        (trajectory[..., :2], yaw.unsqueeze(-1), speed.unsqueeze(-1)), dim=-1
    )
    reference = torch.cat((initial[:, None, None, :].expand(b, k, 1, 4), future), dim=2)
    current, nxt = reference[:, :, :-1], reference[:, :, 1:]
    acceleration = (nxt[..., 3] - current[..., 3]) / dt
    yaw_delta = _wrap(nxt[..., 2] - current[..., 2])
    steering = torch.atan2(yaw_delta * wheelbase, current[..., 3].clamp(min=1e-3) * dt)
    reference_control = torch.stack(
        (acceleration.clamp(-6.0, 4.0), steering.clamp(-0.6, 0.6)), dim=-1
    )
    eye4 = torch.eye(4, dtype=trajectory.dtype, device=trajectory.device)
    q = torch.diag(trajectory.new_tensor([1.0, 4.0, 2.0, 1.0]))
    r = torch.diag(trajectory.new_tensor([0.5, 1.0]))
    p = q.expand(b, k, 4, 4).clone()
    gains = [None] * horizon
    for t in range(horizon - 1, -1, -1):
        ref = current[:, :, t]
        steer = reference_control[:, :, t, 1]
        v, theta = ref[..., 3], ref[..., 2]
        a = eye4.expand(b, k, 4, 4).clone()
        a[..., 0, 2] = -v * theta.sin() * dt
        a[..., 0, 3] = theta.cos() * dt
        a[..., 1, 2] = v * theta.cos() * dt
        a[..., 1, 3] = theta.sin() * dt
        a[..., 2, 3] = steer.tan() * dt / wheelbase
        control_matrix = trajectory.new_zeros((b, k, 4, 2))
        control_matrix[..., 3, 0] = dt
        control_matrix[..., 2, 1] = (
            v * dt / (wheelbase * steer.cos().square().clamp(min=1e-4))
        )
        bt = control_matrix.transpose(-1, -2)
        gain = torch.linalg.solve(r + bt @ p @ control_matrix, bt @ p @ a)
        gains[t] = gain
        p = q + a.transpose(-1, -2) @ p @ (a - control_matrix @ gain)
    state = reference[:, :, 0]
    states = [state]
    for t in range(horizon):
        error = state - reference[:, :, t]
        error = torch.cat((error[..., :2], _wrap(error[..., 2:3]), error[..., 3:]), -1)
        control = reference_control[:, :, t] - (gains[t] @ error.unsqueeze(-1)).squeeze(
            -1
        )
        control = torch.stack(
            (control[..., 0].clamp(-6.0, 4.0), control[..., 1].clamp(-0.6, 0.6)), -1
        )
        state = bicycle_step(state, control, dt, wheelbase)
        states.append(state)
    return torch.stack(states, dim=2)


def make_rollout_evaluator(initial, scorer, dt=0.1, wheelbase=2.7):
    """Adapt a rollout scorer to `planner.select_trajectory`'s evaluator API.

    Scorer receives rollout states [B,K,T+1,4] and a valid-candidate mask
    [B,K] and must return finite [B,K] scores for valid candidates.
    """

    def evaluate(proposals, valid):
        rollouts = lqr_rollout(initial, proposals, dt=dt, wheelbase=wheelbase)
        return scorer(rollouts, valid)

    return evaluate


In [ ]:
initial = torch.tensor([[0., 0., 0., 1.]])
reference = torch.zeros(1, 1, 4, 6)
reference[0, 0, :, 0] = torch.arange(1., 5.) * 0.1
reference[..., 2] = 1.
reference[..., 4] = 1.
rollout = lqr_rollout(initial, reference)
print('tracked x (m):', rollout[0, 0, :, 0])


## Checkpoint workflow

The separate `pluto.train` module implements an epoch-boundary synthetic checkpoint containing model, optimizer, scheduler, RNG state and config. Exact resume is only tested for CPU single-process synthetic data. Optional local MLflow tracking is implemented; raw nuPlan integration and full hosted CI remain future work.


